# Chương 10. Hồi quy tuyến tính

Notebook đi kèm sách *Hành trình AI Engineer*. Mỗi ô code ứng với một mục trong sách, hãy đọc phần giải thích trong sách trước rồi chạy ô tương ứng để kiểm chứng.

Cách chạy: bấm vào từng ô rồi nhấn **Shift + Enter**, hoặc chọn menu **Runtime > Run all**. Chương này không cần GPU.

## 10.4 Code: gradient descent tự cài bằng NumPy

Dữ liệu là bảng 10.1: số giờ tự học mỗi tuần và điểm thi của 4 bạn sinh viên.

In [ ]:
import numpy as np  # NumPy: thư viện tính toán trên mảng số, nhanh hơn vòng lặp Python thuần hàng trăm lần

# Dữ liệu bảng 10.1. dtype=float vì ta sẽ nhân chia với số thập phân
x = np.array([1, 2, 3, 4], dtype=float)  # giờ tự học mỗi tuần của 4 bạn
y = np.array([2, 4, 5, 7], dtype=float)  # điểm thi tương ứng
n = len(x)                               # số mẫu dữ liệu, ở đây n = 4

w, b = 0.0, 0.0  # khởi tạo: chưa biết gì nên bắt đầu từ đường thẳng y = 0
lr = 0.05        # learning rate η, xem khung "Vì sao lại là η = 0,05?"

for buoc in range(201):        # lặp 201 lần để in được cả bước 0 và bước 200
    y_hat = w * x + b          # forward: dự đoán cho CẢ 4 bạn cùng lúc (phép toán vector)
    e = y - y_hat              # phần dư e_i = y_i - ŷ_i, là một mảng 4 phần tử
    loss = np.mean(e ** 2)     # MSE, công thức (10.1)
    if buoc % 50 == 0:         # cứ 50 bước in ra một lần để theo dõi
        print(f"Bước {buoc:3d} | w = {w:.4f} | b = {b:.4f} | MSE = {loss:.6f}")
    grad_w = -2 / n * np.sum(x * e)  # công thức (10.2): x * e nhân TỪNG CẶP phần tử
    grad_b = -2 / n * np.sum(e)      # công thức (10.3)
    w = w - lr * grad_w              # công thức (10.5): đi ngược hướng gradient
    b = b - lr * grad_b

### Kiểm tra lại bằng scikit-learn

Thư viện `scikit-learn` giải bằng nghiệm đóng (công thức 10.4). Kết quả phải trùng với gradient descent ở trên: w = 1,6 và b = 0,5.

In [ ]:
from sklearn.linear_model import LinearRegression  # mô hình hồi quy tuyến tính có sẵn

X = x.reshape(-1, 1)            # sklearn cần X dạng ma trận (n mẫu, d đặc trưng), nên đổi từ (4,) thành (4, 1)
mo_hinh = LinearRegression()    # tạo mô hình rỗng, chưa học gì
mo_hinh.fit(X, y)               # "học": tìm w, b làm MSE nhỏ nhất
print("w =", mo_hinh.coef_[0])          # coef_ là mảng các w, ở đây chỉ có 1 đặc trưng
print("b =", mo_hinh.intercept_)        # intercept_ là hệ số chặn b
print("R² =", mo_hinh.score(X, y))      # score trả về R², sách tính tay được 0,985

## 10.5.2 Thử nghiệm: đổi learning rate

Đây là hình 10.3 trong sách. Hãy thử sửa các giá trị trong danh sách `cac_lr` rồi chạy lại để tự thấy ngưỡng phát nổ (sách tính được là khoảng 0,12).

In [ ]:
import matplotlib.pyplot as plt  # thư viện vẽ biểu đồ

def chay_gd(lr, so_buoc=40):
    """Chạy gradient descent từ w = b = 0, trả về danh sách MSE ở từng bước."""
    w = b = 0.0
    lich_su = []                          # lưu MSE mỗi bước để vẽ
    for _ in range(so_buoc):
        e = y - (w * x + b)               # phần dư
        lich_su.append(np.mean(e ** 2))   # ghi lại MSE trước khi cập nhật
        w -= lr * (-2 / n * np.sum(x * e))  # cập nhật w, viết gọn của w = w - lr * grad_w
        b -= lr * (-2 / n * np.sum(e))      # cập nhật b
    return lich_su

cac_lr = [0.005, 0.05, 0.135]  # quá nhỏ, vừa, quá lớn: thử sửa các số này
for lr_thu in cac_lr:
    plt.plot(chay_gd(lr_thu), label=f"learning rate = {lr_thu}")
plt.yscale("log")              # thang log vì MSE trải từ 0,05 đến hơn 10^9
plt.xlabel("Bước lặp")
plt.ylabel("MSE (thang log)")
plt.legend()
plt.show()